###0. Import Necessary Dependencies for Datasets and Experiments

In [1]:
!pip install -qqq arize arize-phoenix certifi urllib3

In [2]:
import os, certifi
CA = certifi.where()

os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

print("Using CA:", CA)

Using CA: /Users/sean/projects/notebooks/.venv/lib/python3.13/site-packages/certifi/cacert.pem


In [3]:
from typing import Any, Dict
from arize import ArizeClient
from openai import OpenAI
import pandas as pd
from dotenv import load_dotenv

import os
import requests

load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")

In [4]:
client = ArizeClient(api_key=API_KEY, request_verify=False)

In [5]:
examples = client.datasets.list_examples(
    dataset="cnn_dailymail_small",
    space=SPACE_ID,)

  arize.pre_releases | WARNING | [BETA] datasets.list_examples is a beta API in Arize SDK v8.20.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


In [6]:
examples

DatasetsExamplesList200Response(examples=[DatasetExample(id='02f0f515-7050-4589-ad0e-2b15efb13202', created_at=datetime.datetime(2026, 6, 26, 2, 44, 39, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 26, 2, 44, 39, tzinfo=TzInfo(0)), additional_properties={'article': 'WASHINGTON (CNN) -- Vice President Dick Cheney will serve as acting president briefly Saturday while President Bush is anesthetized for a routine colonoscopy, White House spokesman Tony Snow said Friday. Bush is scheduled to have the medical procedure, expected to take about 2 1/2 hours, at the presidential retreat at Camp David, Maryland, Snow said. Bush\'s last colonoscopy was in June 2002, and no abnormalities were found, Snow said. The president\'s doctor had recommended a repeat procedure in about five years. The procedure will be supervised by Dr. Richard Tubb and conducted by a multidisciplinary team from the National Naval Medical Center in Bethesda, Maryland, Snow said. A colonoscopy is the most sensiti

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
chain = llm | StrOutputParser()

prompt = """You are a helpful assistant that summarizes articles.
Summarize the following article in a concise manner:
{article}
"""

def sample_task(dataset_row) -> str:
    # print("Dataset row:", dataset_row)
    article_text = dataset_row["article"]
    return chain.invoke(prompt.format(article=article_text))

In [11]:
from phoenix.evals import llm_classify, OpenAIModel
from arize.experiments import EvaluationResult
import pandas as pd

eval_prompt_template = """
You are comparing the summary text and it's original document and trying to determine if the summary is good. Here is the data:
    [BEGIN DATA]
    ************
    [Summary]: {output}
    ************
    [Original Document]: {input}
    [END DATA]

Compare the Summary above to the Original Document and determine if the Summary is comprehensive, concise, coherent, and independent relative to the Original Document. Your response must be a single word, either "good" or "bad", and should not contain any text or characters aside from that. "bad" means that the Summary is not comprehensive, concise, coherent, and independent relative to the Original Document. "good" means the Summary is comprehensive, concise, coherent, and independent relative to the Original Document.
"""

def summary_eval(output, dataset_row):
    eval_df = llm_classify(
        dataframe=pd.DataFrame([{"input": dataset_row["article"], "output": output}]),
        template=eval_prompt_template,
        model=OpenAIModel(model="gpt-4o-mini"),
        rails=["good", "bad"],
        provide_explanation=True,
    )

    # Map the eval df to EvaluationResult
    label = eval_df["label"][0]
    score = 1 if label == "good" else 0
    explanation = eval_df["explanation"][0]

    return EvaluationResult(label=label, score=score, explanation=explanation)

In [17]:
experiment, experiment_df = client.experiments.run(
    name="summary-01",
    dataset="cnn_dailymail_small",
    space=SPACE_ID,
    task=sample_task,
    evaluators=[summary_eval],
    # dry_run=True
)

  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |██████████| 100/100 (100.0%) | ⏳ 03:53<00:00 |  2.34s/it

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (06/26/26 12:40 PM +0900)
---------------------------------------
   n_examples  n_runs  n_errors
0         100     100         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.



running experiment evaluations |          | 0/100 (0.0%) | ⏳ 00:00<? | ?it/s/var/folders/3n/gv0dncrs175g36_29rs1d_r00000gn/T/ipykernel_57168/1836533491.py:18: DeprecationWarning: `dataframe` argument is deprecated; use `data` instead
  eval_df = llm_classify(
🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.
llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:01<00:00 |  1.89s/it
running experiment evaluations |          | 1/100 (1.0%) | ⏳ 00:02<03:30 |  2.13s/it🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.
llm_classify |██████████| 1/1 (100.0%) | ⏳ 00:01<00:00 |  1.81s/it
running experiment evaluations |▏         | 2/100 (2.0%) | ⏳ 00:04<03:24 |  2.09s/it🐌!! If running inside a notebook, patch

  arize.experiments.functions | INFO | ✅ All evaluators completed.


In [21]:
eval_df = client.experiments.list_runs(experiment=experiment.id, all=True).to_df()
eval_df.head()

,id,example_id,output,result.trace.id,result.trace.timestamp,eval.summary_eval.score,eval.summary_eval.label,eval.summary_eval.explanation,eval.summary_eval.trace.id,eval.summary_eval.trace.timestamp,result
0,EXP_ID_2d60a6,61e8d316-cc2f-4414-ab20-557c2687409b,"**Summary:**\n\nAs Daniel Radcliffe, star of t...",7ae506ea1835144404d2c0946807abd9,1782444975880,1,good,The summary captures the key points of the ori...,6b2fbb968778e5cbab4d9d6f4b4d337a,1782445209881,"**Summary:**\n\nAs Daniel Radcliffe, star of t..."
1,EXP_ID_4c35b5,2c5cc474-c3ab-40f8-af0e-8687c1d5a9ea,**Summary:**\n\nCNN correspondent Soledad O'Br...,4a27ec6ce44609cd09930cd0506c84a2,1782444978470,1,good,The summary effectively captures the main poin...,565160d69c38e66cd88e386937e582c5,1782445212012,**Summary:**\n\nCNN correspondent Soledad O'Br...
2,EXP_ID_9e3d06,5ed19dbb-4650-422d-ade2-baba76909dbd,**Summary:**\n\nWhen the Minneapolis bridge co...,09e33240d1189e653b2107d0170b7b26,1782444980965,1,good,The summary captures the main events and emoti...,32ad3a446a6cf19171ee8103dedb4293,1782445214068,**Summary:**\n\nWhen the Minneapolis bridge co...
3,EXP_ID_be1734,1a21d9bc-9865-4ad7-9b85-86600a3607a3,**Summary:**\n\nPresident Bush had five small ...,36d3a7535357f502a79d398a5954a018,1782444984954,1,good,The summary effectively captures the key point...,a3a3f6df00c13390d28ea1c9fcaf6921,1782445215853,**Summary:**\n\nPresident Bush had five small ...
4,EXP_ID_326413,c9f5975b-7647-4ed4-81cf-89517c261593,**Summary:**\n\nAtlanta Falcons quarterback Mi...,39d34a9125eaabd9b8f4e4e4bdff2cbe,1782444986838,1,good,The summary captures the key points of the ori...,42437440765442780483d265b4a9bfd0,1782445218229,**Summary:**\n\nAtlanta Falcons quarterback Mi...


In [26]:
bad_df = eval_df[eval_df["eval.summary_eval.label"] == "bad"].head(3)
good_df = eval_df[eval_df["eval.summary_eval.label"] == "good"].head(5)
filtered_df = pd.concat([bad_df, good_df])
filtered_df.head(8)

,id,example_id,output,result.trace.id,result.trace.timestamp,eval.summary_eval.score,eval.summary_eval.label,eval.summary_eval.explanation,eval.summary_eval.trace.id,eval.summary_eval.trace.timestamp,result
12,EXP_ID_5a1bfc,aa39bac6-b9fa-436f-b7d3-718fd7170c5c,"**Summary:**\n\nCarlos Alberto, who scored in ...",2de635b2ec94805d68b44d6d62e977e4,1782445011523,0,bad,The summary captures the main points about Car...,98cbaa4294d2deceddd7bd1a3d3a3b6a,1782445233664,"**Summary:**\n\nCarlos Alberto, who scored in ..."
28,EXP_ID_2b1838,29d55a8f-cdbf-4a0b-b3ae-b644458c1d49,"**Summary:**\n\nYoussif, a 5-year-old Iraqi bo...",f5539eac42d03d7bca54d0dc60ba77eb,1782445041907,0,bad,The summary captures the main events of Youssi...,ea9c5f6ee57c33233cea66c2e50a9a8e,1782445264361,"**Summary:**\n\nYoussif, a 5-year-old Iraqi bo..."
93,EXP_ID_48b5f8,6d47b720-cbaf-463c-b122-f1d78d74455c,**Summary:**\n\nPresident Bush vetoed a $600 b...,3c86e074ac20792790346a07f8ed089b,1782445189619,0,bad,The summary captures the main points of Presid...,9480c28c1ef233327122a543a61d1a86,1782445403277,**Summary:**\n\nPresident Bush vetoed a $600 b...
0,EXP_ID_2d60a6,61e8d316-cc2f-4414-ab20-557c2687409b,"**Summary:**\n\nAs Daniel Radcliffe, star of t...",7ae506ea1835144404d2c0946807abd9,1782444975880,1,good,The summary captures the key points of the ori...,6b2fbb968778e5cbab4d9d6f4b4d337a,1782445209881,"**Summary:**\n\nAs Daniel Radcliffe, star of t..."
1,EXP_ID_4c35b5,2c5cc474-c3ab-40f8-af0e-8687c1d5a9ea,**Summary:**\n\nCNN correspondent Soledad O'Br...,4a27ec6ce44609cd09930cd0506c84a2,1782444978470,1,good,The summary effectively captures the main poin...,565160d69c38e66cd88e386937e582c5,1782445212012,**Summary:**\n\nCNN correspondent Soledad O'Br...
2,EXP_ID_9e3d06,5ed19dbb-4650-422d-ade2-baba76909dbd,**Summary:**\n\nWhen the Minneapolis bridge co...,09e33240d1189e653b2107d0170b7b26,1782444980965,1,good,The summary captures the main events and emoti...,32ad3a446a6cf19171ee8103dedb4293,1782445214068,**Summary:**\n\nWhen the Minneapolis bridge co...
3,EXP_ID_be1734,1a21d9bc-9865-4ad7-9b85-86600a3607a3,**Summary:**\n\nPresident Bush had five small ...,36d3a7535357f502a79d398a5954a018,1782444984954,1,good,The summary effectively captures the key point...,a3a3f6df00c13390d28ea1c9fcaf6921,1782445215853,**Summary:**\n\nPresident Bush had five small ...
4,EXP_ID_326413,c9f5975b-7647-4ed4-81cf-89517c261593,**Summary:**\n\nAtlanta Falcons quarterback Mi...,39d34a9125eaabd9b8f4e4e4bdff2cbe,1782444986838,1,good,The summary captures the key points of the ori...,42437440765442780483d265b4a9bfd0,1782445218229,**Summary:**\n\nAtlanta Falcons quarterback Mi...


In [33]:
meta_prompt = """
You are an expert in prompt optimization. You are given an ORIGINAL summarization prompt and PERFORMANCE DATA produced by running that prompt over a dataset. Each performance record contains the model's actual output and an LLM-judge evaluation (`label` = good/bad) with an `eval_explanation` describing WHY the summary was judged good or bad. Use this signal to rewrite the original prompt so that future summaries earn the "good" label.

ORIGINAL BASELINE PROMPT
========================
You are a helpful assistant that summarizes articles.
Summarize the following article in a concise manner:

========================

PERFORMANCE DATA (8 records: 3 bad, 5 good for contrast)
================
{feedback_samples}
================

HOW TO USE THIS DATA
1. Read every `eval_explanation`. The judge states concrete reasons a summary failed or passed.
2. Find the recurring failure pattern across the `bad` records — what do their explanations have
   in common? (e.g. missing specific context, omitted nuances, insufficient comprehensiveness.)
3. Contrast with the `good` records to infer what the judge rewards.
4. Translate those patterns into general, reusable instructions in the prompt — NOT fixes for
   these specific articles.

ALIGNMENT STRATEGY (derive from the explanations, do not hard-code these)
- If summaries are penalized for omitting important context/nuance → instruct the model to
  preserve key context, relationships, and consequential details from the source.
- If summaries are penalized for being too concise / not comprehensive → set explicit coverage
  expectations (capture all major points, not just the headline event).
- Keep whatever the `good` explanations praise (coherence, conciseness) intact.

RULES
Maintain Structure:
- Preserve every template variable from the original prompt verbatim ({{var}}).
- Do not change sections that are already working.
- Preserve the exact output/return-format instructions from the original prompt.

Avoid Overfitting:
- DO NOT copy any article text, summary, or explanation from the performance data into the prompt.
- DO NOT reference these specific examples (Carlos Alberto, Youssif, Bush veto, etc.).
- INSTEAD extract the ESSENCE of what separates good from bad summaries and express it as
  general principles. If you add few-shot examples, make them SYNTHETIC.

Goal: a prompt that generalizes to new documents, not one that memorizes this data.

OUTPUT FORMAT
Return the revised prompt as a JSON array of messages:
[
  {{"role": "system", "content": "..."}},
  {{"role": "user", "content": "..."}}
]

Then add a short "Reasoning" section (bulleted) explaining:
- The dominant failure pattern you found in the `bad` explanations
- Each change you made and which explanation(s) motivated it
"""

In [38]:
import json

response = chain.invoke(
    meta_prompt.format(
        feedback_samples=filtered_df[["output", "eval.summary_eval.label", "eval.summary_eval.explanation"]].to_string(index=False)
    )
)

# The chain returns plain text: a JSON array of messages, then a "Reasoning" section.
# raw_decode parses the leading JSON array and ignores the trailing prose.
start = response.index("[")
optimized_messages, end = json.JSONDecoder().raw_decode(response[start:])
reasoning = response[start + end:].strip()

print(reasoning)
optimized_messages

Reasoning

- Dominant failure pattern in the bad explanations:
  - Summaries were penalized for omitting important context, background, or consequential details, resulting in a lack of comprehensiveness or nuance.
  - The summaries were sometimes too concise, missing key relationships, background, or implications necessary for a full understanding.
- Changes made:
  - Added explicit instructions to capture all major points, key events, and significant relationships or consequences, addressing the lack of comprehensiveness noted in the bad explanations.
  - Instructed to preserve essential background and context, motivated by repeated feedback that missing these aspects led to a 'bad' label.
  - Warned against omitting consequential information or oversimplifying, directly targeting the main reason for summary failures.
  - Retained praise-worthy aspects from the good explanations: conciseness, coherence, independence from the original text, and clarity.
  - Emphasized balancing concise

[{'role': 'system',
  'content': "You are a helpful assistant that summarizes articles. Your summaries should be concise, coherent, and stand independently from the original text. However, do not sacrifice important context or nuance for brevity. Ensure your summary:\n\n- Captures all major points, key events, and significant relationships or consequences described in the article.\n- Preserves essential background, context, and any important details that contribute to a comprehensive understanding of the article's subject.\n- Avoids omitting consequential information or oversimplifying complex situations.\n- Remains clear, well-structured, and free of unnecessary details or direct copying from the original text.\n\nStrive for a balance between conciseness and comprehensiveness so the summary fully represents the article's main ideas and context."},
 {'role': 'user',
  'content': 'Summarize the following article in a concise manner:\n\n{article}'}]